In [ ]:
!pip install datasets soundfile evaluate jiwer
!pip install -U "huggingface_hub[cli]"
!pip install transformers==4.45.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `common_voice_token` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: 

In [ ]:
from datasets import load_dataset
import re
import json
from transformers import Wav2Vec2Processor, Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2ForCTC, TrainingArguments, Trainer, AutoModelForCTC, Wav2Vec2Processor
import IPython.display as ipd
import numpy as np
import random
from dataclasses import dataclass, field
from typing import List, Dict, Union, Optional
import torch
from evaluate import load

In [ ]:
common_voice_asante_twi_train = load_dataset("Lagyamfi/akan_audio_processed", split='train')
common_voice_asante_twi_test = load_dataset("Lagyamfi/akan_audio_processed", split='test')

In [ ]:
chars_to_ignore_regex = '[−\̂\)\‘\-\—\°\‚\¿\”\®\_\￼\´\՚\’\(\*\}\»\&\.\`\'\“\~\$\•\"\{\=\×\̧\̈\:\«\%\+\̀\‑\́\,\→\‹\\\�\…\„\/\›\¡\@\!\–\·\;\?\‐]'

def remove_special_characters(batch):
    batch["sentence"] = re.sub(chars_to_ignore_regex, '', batch["sentence"]).lower() + " "
    return batch

common_voice_asante_twi_train = common_voice_asante_twi_train.map(remove_special_characters)
common_voice_asante_twi_test = common_voice_asante_twi_test.map(remove_special_characters)



Map:   0%|          | 0/2187 [00:00<?, ? examples/s]

Map:   0%|          | 0/259 [00:00<?, ? examples/s]

In [ ]:
def extract_all_chars(batch):
    all_text = " ".join(batch["sentence"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

vocab_train = common_voice_asante_twi_train.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=common_voice_asante_twi_train.column_names)
vocab_test = common_voice_asante_twi_train.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=common_voice_asante_twi_test.column_names)
vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
print(vocab_list)

vocab_dict = {v: k for k, v in enumerate(vocab_list)}
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

with open('vocab.json', 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)



Map:   0%|          | 0/2187 [00:00<?, ? examples/s]

Map:   0%|          | 0/2187 [00:00<?, ? examples/s]

['w', 'h', 'a', 'g', ' ', 'n', 'o', 'k', 'l', 'm', 'ɔ', 'r', 'u', 'ɛ', 's', 't', 'd', 'b', 'f', 'p', 'i', 'e', 'y']


In [ ]:
print(len(vocab_list))

23


In [ ]:
repo_name = "wav2vec2-large-xls-r-300m-asante-twi"

In [ ]:
tokenizer = Wav2Vec2CTCTokenizer("./vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")
tokenizer.push_to_hub(repo_name)
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)


No files have been modified since last commit. Skipping to prevent empty commit.


In [ ]:
rand_int = random.randint(0, len(common_voice_asante_twi_train)-1)

print(common_voice_asante_twi_train[rand_int]["sentence"])
ipd.Audio(data=common_voice_asante_twi_train[rand_int]["audio"]["array"], autoplay=True, rate=16000)

ɔka ɛpo nsuo boa ano kuo 


In [ ]:
rand_int = random.randint(0, len(common_voice_asante_twi_train)-1)

print("Target text:", common_voice_asante_twi_train[rand_int]["sentence"])
print("Input array shape:", common_voice_asante_twi_train[rand_int]["audio"]["array"].shape)
print("Sampling rate:", common_voice_asante_twi_train[rand_int]["audio"]["sampling_rate"])

Target text: na deɛ akyea no bɛtene na akwan mmonkyimmɔnka ayɛ tonomtonom 
Input array shape: (148830,)
Sampling rate: 16000


In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    # batched output is "un-batched"
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["input_length"] = len(batch["input_values"])

    with processor.as_target_processor():
        batch["labels"] = processor(batch["sentence"]).input_ids
    return batch

train_dataset = common_voice_asante_twi_train.map(prepare_dataset, remove_columns=common_voice_asante_twi_train.column_names)
eval_dataset = common_voice_asante_twi_test.map(prepare_dataset, remove_columns=common_voice_asante_twi_test.column_names)


Map:   0%|          | 0/2187 [00:00<?, ? examples/s]

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lenghts and need
        # different padding methods
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels

        return batch


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.0,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
).to(device)


model.freeze_feature_extractor()

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-300m and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/modeling_wav2vec2.py:2177: FutureWarning: The method `freeze_feature_extractor` is deprecated and will be removed in Transformers v5. Please use the equivalent `freeze_feature_encoder` method instead.
  warnings.warn(


In [ ]:
repo_name = "wav2vec2-large-xls-r-1b-asante-twi"

In [ ]:
 training_args = TrainingArguments(
    output_dir=repo_name,
    group_by_length=True,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    evaluation_strategy="steps",
    num_train_epochs=40,
    gradient_checkpointing=True,
    fp16=True,
    save_steps=400,
    eval_steps=400,
    logging_steps=400,
    learning_rate=3e-4,
    warmup_steps=500,
    save_total_limit=2,
    push_to_hub=True,
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
wer_metric = load("wer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}


In [ ]:
# Create an instance of the data collator
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# Create the trainer with the data_collator
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=processor.feature_extractor
)

trainer.train()


Step,Training Loss,Validation Loss,Wer
400,3.451200,0.534656,0.622306
800,0.256800,0.431363,0.494845
1200,0.105300,0.466642,0.456420
1600,0.057900,0.491558,0.433927
2000,0.039500,0.489438,0.420337
2400,0.027400,0.494429,0.411434


/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:157: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:157: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:157: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call

TrainOutput(global_step=2720, training_loss=0.5814243293860379, metrics={'train_runtime': 5010.0395, 'train_samples_per_second': 17.461, 'train_steps_per_second': 0.543, 'total_flos': 1.6882689025449935e+19, 'train_loss': 0.5814243293860379, 'epoch': 39.70802919708029})

In [ ]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/ransfordnyarko/wav2vec2-large-xls-r-300m-asante-twi/commit/20abb63c172b91b454631ebceb1653c5368306a4', commit_message='End of training', commit_description='', oid='20abb63c172b91b454631ebceb1653c5368306a4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ransfordnyarko/wav2vec2-large-xls-r-300m-asante-twi', endpoint='https://huggingface.co', repo_type='model', repo_id='ransfordnyarko/wav2vec2-large-xls-r-300m-asante-twi'), pr_revision=None, pr_num=None)

In [ ]:
model = AutoModelForCTC.from_pretrained(f"ransfordnyarko/{repo_name}")
processor = Wav2Vec2Processor.from_pretrained(f"ransfordnyarko/{repo_name}")

config.json:   0%|          | 0.00/2.09k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(f"ransfordnyarko/{repo_name}")


In [ ]:
processor = Wav2Vec2Processor.from_pretrained(f"ransfordnyarko/{repo_name}")

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    # batched output is "un-batched"
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["input_length"] = len(batch["input_values"])

    with processor.as_target_processor():
        batch["labels"] = processor(batch["sentence"]).input_ids
    return batch

In [ ]:
input_dict = processor(eval_dataset[2]["input_values"], return_tensors="pt", padding=True)

logits = model(input_dict.input_values).logits

pred_ids = torch.argmax(logits, dim=-1)[0]

It is strongly recommended to pass the ``sampling_rate`` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


In [ ]:
common_voice_asante_twi_test = load_dataset("Lagyamfi/akan_audio_processed", split='test')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/3.33k [00:00<?, ?B/s]

(…)-00000-of-00001-2c875d50b966a0c3.parquet:   0%|          | 0.00/427M [00:00<?, ?B/s]

(…)-00000-of-00001-fd53689d8cd93e01.parquet:   0%|          | 0.00/42.5M [00:00<?, ?B/s]

(…)-00000-of-00001-1a99ff63881e72f6.parquet:   0%|          | 0.00/401M [00:00<?, ?B/s]

...

Generating test_Vtlp_Aug split:   0%|          | 0/259 [00:00<?, ? examples/s]

Generating train_Normalize_Aug split:   0%|          | 0/2187 [00:00<?, ? examples/s]

Generating test_Normalize_Aug split:   0%|          | 0/259 [00:00<?, ? examples/s]

Generating train_PolarityInverse_Aug split:   0%|          | 0/2187 [00:00<?, ? examples/s]

Generating test_PolarityInverse_Aug split:   0%|          | 0/259 [00:00<?, ? examples/s]

In [ ]:
processor

Wav2Vec2Processor:
- feature_extractor: Wav2Vec2FeatureExtractor {
  "do_normalize": true,
  "feature_extractor_type": "Wav2Vec2FeatureExtractor",
  "feature_size": 1,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": true,
  "sampling_rate": 16000
}

- tokenizer: Wav2Vec2CTCTokenizer(name_or_path='ransfordnyarko/wav2vec2-large-xls-r-300m-asante-twi', vocab_size=23, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '[UNK]', 'pad_token': '[PAD]'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	23: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	24: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	25: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	26: AddedToken(

In [ ]:
eval_dataset = common_voice_asante_twi_test.map(prepare_dataset, remove_columns=common_voice_asante_twi_test.column_names)

Map:   0%|          | 0/259 [00:00<?, ? examples/s]

In [ ]:
for i in range(0, 5):
  i = int(i)
  input_dict = processor(eval_dataset[i]["input_values"], return_tensors="pt", padding=True)

  logits = model(input_dict.input_values).logits
  pred_ids = torch.argmax(logits, dim=-1)[0]
  print("Prediction:")
  print(processor.decode(pred_ids))

  print("\nReference:")
  print(common_voice_asante_twi_test[i]["sentence"].lower())
  print("-" * 100)

It is strongly recommended to pass the ``sampling_rate`` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the ``sampling_rate`` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


Prediction:
awurade ne mobotan ne mabankɛseɛ ne me gyefoɔ

Reference:
awurade ne me botan ne mabankɛseɛ ne me gyefoɔ 
----------------------------------------------------------------------------------------------------


It is strongly recommended to pass the ``sampling_rate`` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


Prediction:
me nyankopɔn ne me botantim a medwane kɔtoa no

Reference:
me nyankopɔn ne me botantim a medwane kɔtoa no 
----------------------------------------------------------------------------------------------------


It is strongly recommended to pass the ``sampling_rate`` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


Prediction:
me kyɛm ne me nkwagyeɛ abɛn mabantenten

Reference:
me kyɛm ne me nkwagyeɛ abɛn mabantenten 
----------------------------------------------------------------------------------------------------


It is strongly recommended to pass the ``sampling_rate`` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


Prediction:
mesu mefrɛ awurade ɔno a ɔfata sɛ wɔyii no ayɛ

Reference:
mesu mefrɛ awurade ɔno a ɔfata sɛ wɔyi no ayɛ 
----------------------------------------------------------------------------------------------------
Prediction:
na woagye me matamfoɔ nsam

Reference:
na wɔagye me matamfoɔ nsam 
----------------------------------------------------------------------------------------------------


# **Results from xls-r-300m**

### **Prediction**:
awurade ne mobotan ne mabankɛseɛ ne me gyefoɔ

### **Reference**:
awurade ne me botan ne mabankɛseɛ ne me gyefoɔ


---

### **Prediction**:
me nyankopɔn ne me botantim a medwane kɔtoa no

### **Reference**:
me nyankopɔn ne me botantim a medwane kɔtoa no

---

### **Prediction**:
me kyɛm ne me nkwagyeɛ abɛn mabantenten


### **Reference**:
me kyɛm ne me nkwagyeɛ abɛn mabantenten

---

### **Prediction**:
mesu mefrɛ awurade ɔno a ɔfata sɛ wɔyii no ayɛ

### **Reference**:
mesu mefrɛ awurade ɔno a ɔfata sɛ wɔyi no ayɛ

---

### **Prediction**:
na woagye me matamfoɔ nsam

### **Reference**:
na wɔagye me matamfoɔ nsam

In [ ]:
print("Prediction:")
print(processor.decode(pred_ids))

print("\nReference:")
print(common_voice_asante_twi_test[2]["sentence"].lower())

Prediction:
me kyɛm ne me nkwagyeɛ abɛn mabantenten

Reference:
me kyɛm ne me nkwagyeɛ abɛn, m’abantenten.


## Training on Wav2Vec 1B parameters

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    # batched output is "un-batched"
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["input_length"] = len(batch["input_values"])

    with processor.as_target_processor():
        batch["labels"] = processor(batch["sentence"]).input_ids
    return batch

train_dataset = common_voice_asante_twi_train.map(prepare_dataset, remove_columns=common_voice_asante_twi_train.column_names)
eval_dataset = common_voice_asante_twi_test.map(prepare_dataset, remove_columns=common_voice_asante_twi_test.column_names)


NameError: name 'common_voice_asante_twi_train' is not defined

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-1b",
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.0,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
).to(device)


model.freeze_feature_extractor()

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-1b and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/modeling_wav2vec2.py:2177: FutureWarning: The method `freeze_feature_extractor` is deprecated and will be removed in Transformers v5. Please use the equivalent `freeze_feature_encoder` method instead.
  warnings.warn(


In [ ]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# Create the trainer with the data_collator
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=processor.feature_extractor
)

trainer.train()

Step,Training Loss,Validation Loss,Wer
400,0.058700,0.447474,0.449391
800,0.114800,0.467774,0.417526
1200,0.057400,0.413491,0.412371
1600,0.042200,0.453525,0.379100
2000,0.026900,0.468992,0.369260


/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:157: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:157: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:157: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call

Step,Training Loss,Validation Loss,Wer
400,0.058700,0.447474,0.449391
800,0.114800,0.467774,0.417526
1200,0.057400,0.413491,0.412371
1600,0.042200,0.453525,0.379100
2000,0.026900,0.468992,0.369260
2400,0.019800,0.451213,0.363168


/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:157: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(


In [ ]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/ransfordnyarko/wav2vec2-large-xls-r-1b-asante-twi/commit/65104e304b50ba72fe6cc24b96a75c0aeb4ccd20', commit_message='End of training', commit_description='', oid='65104e304b50ba72fe6cc24b96a75c0aeb4ccd20', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ransfordnyarko/wav2vec2-large-xls-r-1b-asante-twi', endpoint='https://huggingface.co', repo_type='model', repo_id='ransfordnyarko/wav2vec2-large-xls-r-1b-asante-twi'), pr_revision=None, pr_num=None)

In [ ]:
repo_name = "wav2vec2-large-xls-r-1b-asante-twi"
processor_name = "wav2vec2-large-xls-r-300m-asante-twi"

In [ ]:
model = AutoModelForCTC.from_pretrained(f"ransfordnyarko/{repo_name}")
processor = Wav2Vec2Processor.from_pretrained(f"ransfordnyarko/{processor_name}")
model = Wav2Vec2ForCTC.from_pretrained(f"ransfordnyarko/{repo_name}")


In [ ]:
processor = Wav2Vec2Processor.from_pretrained(f"ransfordnyarko/{repo_name}")

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    # batched output is "un-batched"
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["input_length"] = len(batch["input_values"])

    with processor.as_target_processor():
        batch["labels"] = processor(batch["sentence"]).input_ids
    return batch

In [ ]:
common_voice_asante_twi_test = load_dataset("Lagyamfi/akan_audio_processed", split='test')


In [ ]:
eval_dataset = common_voice_asante_twi_test.map(prepare_dataset, remove_columns=common_voice_asante_twi_test.column_names)

Map:   0%|          | 0/259 [00:00<?, ? examples/s]

ValueError: text input must be of type `str` (single example), `List[str]` (batch or single pretokenized example) or `List[List[str]]` (batch of pretokenized examples).

In [ ]:
for i in range(0, 5):
  i = int(i)
  input_dict = processor(eval_dataset[i]["input_values"], return_tensors="pt", padding=True)

  logits = model(input_dict.input_values).logits
  pred_ids = torch.argmax(logits, dim=-1)[0]
  print("Prediction:")
  print(processor.decode(pred_ids))

  print("\nReference:")
  print(common_voice_asante_twi_test[i]["sentence"].lower())
  print("-" * 100)

It is strongly recommended to pass the ``sampling_rate`` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the ``sampling_rate`` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


Prediction:
awurade ne mobotan ne mabankɛseɛ ne me gyefoɔ

Reference:
awurade ne me botan ne mabankɛseɛ ne me gyefoɔ 
----------------------------------------------------------------------------------------------------


It is strongly recommended to pass the ``sampling_rate`` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


Prediction:
me nyankopɔn ne me botantim a medwane kɔtoa no

Reference:
me nyankopɔn ne me botantim a medwane kɔtoa no 
----------------------------------------------------------------------------------------------------


It is strongly recommended to pass the ``sampling_rate`` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


Prediction:
me kyɛm ne me nkwagyeɛ abɛn mabantenten

Reference:
me kyɛm ne me nkwagyeɛ abɛn mabantenten 
----------------------------------------------------------------------------------------------------


It is strongly recommended to pass the ``sampling_rate`` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


Prediction:
mesu mefrɛ awurade ɔno a ɔfata sɛ wɔyii no ayɛ

Reference:
mesu mefrɛ awurade ɔno a ɔfata sɛ wɔyi no ayɛ 
----------------------------------------------------------------------------------------------------
Prediction:
na woagye me matamfoɔ nsam

Reference:
na wɔagye me matamfoɔ nsam 
----------------------------------------------------------------------------------------------------


## Audio system


In [ ]:
!pip install scipy wavio sounddevice

In [ ]:
!apt-get install -y portaudio19-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libportaudio2 libportaudiocpp0
Suggested packages:
  portaudio19-doc
The following NEW packages will be installed:
  libportaudio2 libportaudiocpp0 portaudio19-dev
0 upgraded, 3 newly installed, 0 to remove and 49 not upgraded.
Need to get 188 kB of archives.
After this operation, 927 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libportaudio2 amd64 19.6.0-1.1 [65.3 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libportaudiocpp0 amd64 19.6.0-1.1 [16.1 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 portaudio19-dev amd64 19.6.0-1.1 [106 kB]
Fetched 188 kB in 1s (138 kB/s)
Selecting previously unselected package libportaudio2:amd64.
(Reading database ... 123630 files and directories currently installed.)
Preparing to unpack .../libportaudio2_19.6.0-1.

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import numpy as np

In [ ]:


def record_audio(filename, duration=5, sample_rate=16000):

    print("Recording...")
    audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='int16')
    sd.wait()  # Wait until the recording is finished
    print("Recording finished.")
    write(filename, sample_rate, audio)  # Save the audio as a .wav file

# Record 5 seconds of audio and save it as 'output.wav'
record_audio("output.wav", duration=20)


Recording...


PortAudioError: Error querying device -1